
# Klike Data Science Challenge
## Parte 3 — Recommendations Engine

**Empresa:** Millai | Klike  
**Dataset:** `klike_challenge_dataset.csv`  
**Linguagem:** Python  
**Objetivo:** desenvolver um sistema de recomendações acionáveis e contextualizadas para otimização de criativos publicitários.

---

### Contextualização

Na Parte 1, foram investigadas as características das campanhas, a qualidade dos dados e as associações entre atributos dos criativos e métricas de desempenho.

Na Parte 2, foi desenvolvido um modelo de regressão Ridge reduzido, utilizando 10 características disponíveis antes da veiculação das campanhas para estimar o `klike_score`.

O modelo apresentou MAE de 6,685 e R² de 0,667 no conjunto de teste.

Nesta terceira etapa, o objetivo é transformar os resultados exploratórios e preditivos em um **Recommendations Engine**, capaz de analisar uma campanha e identificar oportunidades de melhoria.

O sistema utilizará uma abordagem híbrida, combinando:

- **Modelagem preditiva:** simulação de alterações nas características dos criativos por meio do Ridge reduzido.
- **Evidências estatísticas:** comparação de campanhas com características semelhantes, considerando seu contexto.
- **Priorização:** ordenação das sugestões conforme as diferenças previstas no Klike Score, respeitando a suficiência das evidências disponíveis.

As recomendações serão interpretadas como oportunidades de otimização, e não como garantias de melhoria. Diferenças preditivas e associações observacionais não constituem evidências de causalidade.



### Metodologia do Recommendations Engine

O desenvolvimento será organizado nas seguintes etapas:

1. **Preparação e reprodutibilidade:** carregamento do dataset e recuperação do Ridge reduzido desenvolvido na Parte 2.
2. **Validação das entradas:** identificação das características necessárias para analisar uma campanha e verificação de sua consistência.
3. **Geração de cenários alternativos:** definição de alterações acionáveis nas características dos criativos.
4. **Simulação preditiva:** utilização do Ridge para estimar diferenças no Klike Score entre a campanha original e os cenários alternativos.
5. **Análise contextual:** investigação das associações observadas entre características criativas e desempenho em campanhas comparáveis.
6. **Seleção e priorização:** filtragem de alterações não sustentadas pelas evidências e ordenação das oportunidades identificadas.
7. **Geração das recomendações:** apresentação de sugestões específicas, quantificadas e acompanhadas de suas limitações.
8. **Demonstração e validação:** execução do sistema em campanhas do dataset e verificação da consistência dos resultados.

**Princípios metodológicos:**

- Reutilizar o modelo selecionado na Parte 2, evitando treinamento desnecessário.
- Utilizar exclusivamente os registros de treinamento para construir as evidências estatísticas que fundamentam as recomendações.
- Preservar o conjunto de teste, evitando utilizá-lo na definição ou otimização das regras do engine.
- Não utilizar métricas pós-veiculação como entradas do modelo preditivo.
- Diferenciar explicitamente associações estatísticas, diferenças entre previsões e efeitos causais.
- Evitar recomendações baseadas em grupos com suporte amostral insuficiente.
- Não apresentar melhorias quantitativas sem uma estimativa calculada e devidamente contextualizada.

O sistema será desenvolvido inicialmente como um protótipo funcional em Python, utilizando funções independentes e reproduzíveis.



---

## 1. Preparação do ambiente e recuperação do modelo

### 1.1. Importação das bibliotecas e verificação dos arquivos

Nesta etapa, serão importadas as bibliotecas necessárias para manipulação dos dados, recuperação do modelo e reprodução da divisão entre treinamento e teste.

O Ridge reduzido foi salvo na Parte 2 utilizando `joblib`, juntamente com a lista ordenada das features utilizadas no treinamento.

Antes de carregar os arquivos, será verificada sua disponibilidade no diretório do projeto.

Essa abordagem permite reutilizar o pipeline previamente treinado sem realizar novos ajustes nos parâmetros do modelo.


In [1]:

from pathlib import Path

import joblib
import pandas as pd

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

RANDOM_STATE = 42

# Caminhos dos arquivos
DATA_PATH = Path("klike_challenge_dataset.csv")
MODEL_PATH = Path("models/ridge_klike.joblib")
FEATURES_PATH = Path("models/features_klike.joblib")

# Verificação de disponibilidade
arquivos = {
    "Dataset original": DATA_PATH,
    "Ridge reduzido": MODEL_PATH,
    "Features do modelo": FEATURES_PATH
}

status_arquivos = pd.DataFrame([
    {
        "Arquivo": nome,
        "Caminho": str(caminho),
        "Disponível": caminho.is_file()
    }
    for nome, caminho in arquivos.items()
])

display(status_arquivos)


,Arquivo,Caminho,Disponível
0,Dataset original,klike_challenge_dataset.csv,True
1,Ridge reduzido,models\ridge_klike.joblib,True
2,Features do modelo,models\features_klike.joblib,True



### 1.2. Carregamento dos dados e reconstrução da divisão treino/teste

O dataset será carregado sem alterações nos registros originais.

Em seguida, serão recuperados o pipeline Ridge treinado e a lista de features utilizada na Parte 2.

Para preservar a reprodutibilidade, será reconstruída a mesma divisão aleatória entre treinamento e teste, utilizando `test_size=0.20` e `random_state=42`.

As variáveis booleanas serão convertidas para a representação numérica adotada anteriormente, preservando os valores ausentes para tratamento pelo próprio pipeline.

A reconstrução da divisão é necessária porque as estatísticas contextuais do Recommendations Engine serão calculadas exclusivamente sobre as campanhas de treinamento.

O conjunto de teste permanecerá separado e não será utilizado na elaboração das recomendações.


In [2]:

# Verificar os arquivos necessários
arquivos_ausentes = [
    str(caminho)
    for caminho in arquivos.values()
    if not caminho.is_file()
]

if arquivos_ausentes:
    raise FileNotFoundError(
        f"Arquivos não encontrados: {arquivos_ausentes}"
    )

# Carregar dataset e artefatos da Parte 2
df = pd.read_csv(DATA_PATH)

modelo_ridge = joblib.load(MODEL_PATH)
features_reduzidas = joblib.load(FEATURES_PATH)

# Features definidas na Parte 2
features_esperadas = [
    "platform",
    "category",
    "is_retargeting",
    "video_duration_s",
    "format",
    "has_subtitle",
    "has_cta",
    "has_hook",
    "has_face",
    "text_density"
]

assert features_reduzidas == features_esperadas, (
    "As features carregadas diferem da Parte 2."
)

assert isinstance(modelo_ridge, Pipeline), (
    "O modelo carregado não é um Pipeline."
)

assert isinstance(modelo_ridge.named_steps["model"], Ridge), (
    "O estimador carregado não é um Ridge."
)

assert modelo_ridge.named_steps["model"].alpha == 1.0, (
    "O alpha do Ridge difere da Parte 2."
)

assert df["klike_score"].notna().all(), (
    "O dataset contém valores ausentes no target."
)

assert df["campaign_id"].is_unique, (
    "Existem identificadores de campanha duplicados."
)

# Preparar as mesmas entradas do Ridge reduzido
X = df[features_reduzidas].copy()
y = df["klike_score"].copy()

colunas_booleanas = [
    "is_retargeting",
    "has_subtitle",
    "has_cta",
    "has_hook",
    "has_face"
]

for coluna in colunas_booleanas:
    X[coluna] = X[coluna].map({
        True: 1,
        False: 0
    }).astype(float)

# Reproduzir a divisão utilizada na Parte 2
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

# Registros originais de treinamento para análises contextuais
df_train = df.loc[X_train.index].copy()

# Resumo da preparação
print("Modelo:", type(modelo_ridge.named_steps["model"]).__name__)
print("Alpha:", modelo_ridge.named_steps["model"].alpha)
print("Etapas do pipeline:", list(modelo_ridge.named_steps.keys()))

print("\nDimensões:")
print(f"Dataset original: {df.shape}")
print(f"Treinamento: {X_train.shape}")
print(f"Teste: {X_test.shape}")
print(f"Features utilizadas: {len(features_reduzidas)}")

print("\nFeatures do modelo:")
for i, feature in enumerate(features_reduzidas, start=1):
    print(f"{i:2d}. {feature}")

print("\nValidação dos índices:")
print(
    "Interseção treino/teste:",
    len(set(X_train.index) & set(X_test.index))
)
print(
    "Registros para evidência contextual:",
    len(df_train)
)


Modelo: Ridge
Alpha: 1.0
Etapas do pipeline: ['preprocessor', 'model']

Dimensões:
Dataset original: (500, 26)
Treinamento: (400, 10)
Teste: (100, 10)
Features utilizadas: 10

Features do modelo:
 1. platform
 2. category
 3. is_retargeting
 4. video_duration_s
 5. format
 6. has_subtitle
 7. has_cta
 8. has_hook
 9. has_face
10. text_density

Validação dos índices:
Interseção treino/teste: 0
Registros para evidência contextual: 400



---

## 2. Validação das entradas e definição das características modificáveis

### 2.1. Classificação das features

O Recommendations Engine utilizará as mesmas 10 features do Ridge reduzido, divididas em dois grupos:

**Características de contexto:** descrevem o ambiente e o direcionamento da campanha. Serão mantidas constantes durante as simulações para que as recomendações sejam contextualizadas.

**Características modificáveis:** representam aspectos do criativo que podem ser alterados para produzir cenários alternativos.

A classificação é importante para impedir que o sistema recomende alterações em características que não correspondem diretamente à otimização do criativo.

Nesta etapa, também serão verificados os valores categóricos presentes no conjunto de treinamento e a quantidade de registros ausentes.

As categorias conhecidas serão utilizadas para validar os inputs. Essa validação evita que valores desconhecidos sejam silenciosamente codificados pelo pipeline, comprometendo a interpretação das previsões.


In [3]:

# Características que representam o contexto da campanha
features_contexto = [
    "platform",
    "category",
    "is_retargeting"
]

# Características que podem ser modificadas no criativo
features_modificaveis = [
    "video_duration_s",
    "format",
    "has_subtitle",
    "has_cta",
    "has_hook",
    "has_face",
    "text_density"
]

# Classificação por tipo de dado
features_categoricas = [
    "platform",
    "category",
    "format",
    "text_density"
]

features_booleanas = [
    "is_retargeting",
    "has_subtitle",
    "has_cta",
    "has_hook",
    "has_face"
]

features_numericas = [
    "video_duration_s"
]

# Verificar se todas as features foram classificadas
assert set(features_contexto).isdisjoint(
    features_modificaveis
)

assert set(features_contexto + features_modificaveis) == set(
    features_reduzidas
)

assert set(
    features_categoricas + features_booleanas + features_numericas
) == set(features_reduzidas)

# Resumo das características
resumo_features = pd.DataFrame({
    "Feature": features_reduzidas,
    "Papel": [
        "Contexto" if feature in features_contexto
        else "Modificável"
        for feature in features_reduzidas
    ],
    "Tipo": [
        "Categórica" if feature in features_categoricas
        else "Booleana" if feature in features_booleanas
        else "Numérica"
        for feature in features_reduzidas
    ],
    "Valores ausentes no treino": [
        df_train[feature].isna().sum()
        for feature in features_reduzidas
    ]
})

display(resumo_features)

# Categorias efetivamente observadas no treinamento
categorias_validas = {
    feature: sorted(df_train[feature].dropna().unique().tolist())
    for feature in features_categoricas
}

print("\nCategorias disponíveis no treinamento:")

for feature, categorias in categorias_validas.items():
    print(f"{feature}: {categorias}")

print("\nDistribuição da duração do vídeo no treinamento:")
display(
    df_train["video_duration_s"]
    .describe()
    .to_frame(name="video_duration_s")
)


,Feature,Papel,Tipo,Valores ausentes no treino
0,platform,Contexto,Categórica,0
1,category,Contexto,Categórica,0
2,is_retargeting,Contexto,Booleana,0
3,video_duration_s,Modificável,Numérica,34
4,format,Modificável,Categórica,0
5,has_subtitle,Modificável,Booleana,38
6,has_cta,Modificável,Booleana,0
7,has_hook,Modificável,Booleana,0
8,has_face,Modificável,Booleana,0
9,text_density,Modificável,Categórica,0



Categorias disponíveis no treinamento:
platform: ['LinkedIn', 'Meta', 'TikTok']
category: ['App Install', 'Branding', 'E-commerce', 'Lead Gen', 'SaaS']
format: ['horizontal', 'quadrado', 'vertical']
text_density: ['high', 'low', 'medium']

Distribuição da duração do vídeo no treinamento:


,video_duration_s
count,366.000000
mean,19.702186
std,14.168704
min,5.000000
25%,11.000000
50%,16.000000
75%,26.000000
max,115.000000



### 2.2. Preparação e validação de uma campanha individual

O protótipo receberá uma campanha representada por um dicionário Python ou por uma linha do dataset (`pandas.Series`).

A função de preparação terá as seguintes responsabilidades:

- Verificar se as 10 features necessárias estão presentes.
- Validar os valores categóricos utilizando apenas as categorias observadas no treinamento.
- Converter características booleanas para a representação numérica utilizada no Ridge.
- Verificar se a duração do vídeo é numericamente válida.
- Preservar valores ausentes nas características que já possuíam ausências no dataset e foram tratadas pelo pipeline.
- Organizar as características na ordem esperada pelo modelo.

O processo não utilizará `klike_score`, CTR, ROAS ou outras métricas pós-veiculação como entradas preditivas.

A função desenvolvida nesta seção servirá como interface de entrada para o Recommendations Engine, permitindo analisar campanhas existentes ou novas campanhas informadas manualmente.


In [4]:

import numpy as np

def preparar_campanha(campanha):
    """
    Valida e prepara uma campanha para o Ridge reduzido.

    Parâmetros
    ----------
    campanha : dict ou pd.Series
        Características de uma campanha.

    Retorna
    -------
    pd.DataFrame
        DataFrame com uma linha e as 10 features
        na ordem esperada pelo modelo.
    """

    if isinstance(campanha, pd.Series):
        campanha = campanha.to_dict()

    if not isinstance(campanha, dict):
        raise TypeError(
            "A campanha deve ser um dict ou pd.Series."
        )

    features_ausentes = [
        feature
        for feature in features_reduzidas
        if feature not in campanha
    ]

    if features_ausentes:
        raise ValueError(
            f"Features ausentes: {features_ausentes}"
        )

    entrada = {
        feature: campanha[feature]
        for feature in features_reduzidas
    }

    # Validar variáveis categóricas
    for feature in features_categoricas:
        valor = entrada[feature]

        if pd.isna(valor) or valor not in categorias_validas[feature]:
            raise ValueError(
                f"Valor inválido para {feature}: {valor}. "
                f"Valores aceitos: {categorias_validas[feature]}"
            )

    # Converter booleanos para 0 ou 1
    for feature in features_booleanas:
        valor = entrada[feature]

        if pd.isna(valor):
            if feature == "has_subtitle":
                entrada[feature] = np.nan
                continue

            raise ValueError(
                f"A feature {feature} não pode ser nula."
            )

        if isinstance(valor, (bool, np.bool_)):
            entrada[feature] = int(valor)

        elif isinstance(valor, (int, float, np.integer, np.floating)) \
                and valor in (0, 1):
            entrada[feature] = int(valor)

        else:
            raise ValueError(
                f"{feature} deve ser True, False, 0 ou 1."
            )

    # Validar a duração do vídeo
    duracao = entrada["video_duration_s"]

    if pd.isna(duracao):
        entrada["video_duration_s"] = np.nan

    elif (
        isinstance(duracao, (bool, np.bool_))
        or not isinstance(
            duracao,
            (int, float, np.integer, np.floating)
        )
        or not np.isfinite(duracao)
        or duracao <= 0
    ):
        raise ValueError(
            "video_duration_s deve ser um número positivo "
            "e finito, ou um valor ausente."
        )

    else:
        entrada["video_duration_s"] = float(duracao)

    return pd.DataFrame(
        [entrada],
        columns=features_reduzidas
    )


# Selecionar uma campanha do conjunto de treinamento
campanha_exemplo = df_train.iloc[0]

# Preparar o input
entrada_validada = preparar_campanha(campanha_exemplo)

print("Campanha selecionada:")
print(campanha_exemplo["campaign_id"])

print("\nInput validado para o Recommendations Engine:")
display(entrada_validada)


Campanha selecionada:
KLK-0250

Input validado para o Recommendations Engine:


,platform,category,is_retargeting,video_duration_s,format,has_subtitle,has_cta,has_hook,has_face,text_density
0,LinkedIn,App Install,0,22.0,vertical,1,1,1,0,medium



---

## 3. Geração de cenários alternativos

### 3.1. Definição das alterações candidatas

O Recommendations Engine avaliará possíveis modificações nas características dos criativos, preservando o contexto original de cada campanha.

Cada cenário será gerado alterando **uma única feature por vez**, permitindo comparar a previsão da campanha original com a previsão de uma configuração alternativa.

As modificações serão definidas conforme o tipo de variável:

- **Booleanas:** testar a inclusão de características ausentes, como hook, CTA, legendas e rosto humano.
- **Categóricas:** testar formatos e densidades de texto existentes no conjunto de treinamento.
- **Numéricas:** testar durações alternativas baseadas nos quartis observados em campanhas da mesma plataforma, sem assumir que esses valores sejam ideais.

As características `platform`, `category` e `is_retargeting` permanecerão constantes durante as simulações.

Valores ausentes não serão interpretados automaticamente como ausência de uma característica. Nesses casos, não serão geradas alterações para a feature até que seu valor original seja conhecido.

Os cenários representam **hipóteses de melhoria**, não recomendações finais. Sua seleção dependerá posteriormente das diferenças previstas pelo Ridge e das evidências estatísticas contextuais.

Para evitar data leakage, as alternativas baseadas na distribuição dos dados serão calculadas exclusivamente a partir do conjunto de treinamento.


In [5]:

def gerar_cenarios(campanha):
    """
    Gera cenários alternativos modificando uma feature
    por vez, mantendo o contexto original da campanha.

    Não calcula impacto nem classifica melhorias.
    """

    entrada = preparar_campanha(campanha)
    original = entrada.iloc[0].to_dict()

    cenarios = []

    def adicionar_cenario(feature, novo_valor, origem):
        alternativa = original.copy()
        alternativa[feature] = novo_valor

        cenarios.append({
            "feature_alterada": feature,
            "valor_original": original[feature],
            "valor_proposto": novo_valor,
            "origem": origem,
            **alternativa
        })

    # 1. Características booleanas
    booleanas_modificaveis = [
        "has_subtitle",
        "has_cta",
        "has_hook",
        "has_face"
    ]

    for feature in booleanas_modificaveis:
        valor = original[feature]

        if pd.notna(valor) and valor == 0:
            adicionar_cenario(
                feature,
                1,
                "Inclusão de característica ausente"
            )

    # 2. Características categóricas
    categoricas_modificaveis = [
        "format",
        "text_density"
    ]

    for feature in categoricas_modificaveis:
        for valor in categorias_validas[feature]:

            if valor != original[feature]:
                adicionar_cenario(
                    feature,
                    valor,
                    "Categoria observada no treinamento"
                )

    # 3. Duração do vídeo
    duracao_atual = original["video_duration_s"]

    if pd.notna(duracao_atual):

        duracoes_plataforma = df_train.loc[
            df_train["platform"] == original["platform"],
            "video_duration_s"
        ].dropna()

        # Evitar referências baseadas em grupos muito pequenos
        if len(duracoes_plataforma) >= 20:

            quartis = duracoes_plataforma.quantile(
                [0.25, 0.50, 0.75]
            )

            duracoes_candidatas = sorted({
                int(round(valor))
                for valor in quartis
                if valor > 0
            })

            for duracao in duracoes_candidatas:

                if not np.isclose(duracao, duracao_atual):
                    adicionar_cenario(
                        "video_duration_s",
                        duracao,
                        "Quartis da duração por plataforma no treino"
                    )

    # Organizar os cenários
    colunas = [
        "feature_alterada",
        "valor_original",
        "valor_proposto",
        "origem"
    ] + features_reduzidas

    return pd.DataFrame(cenarios, columns=colunas)


# Gerar cenários para a campanha de exemplo
cenarios_exemplo = gerar_cenarios(campanha_exemplo)

print("Campanha:", campanha_exemplo["campaign_id"])
print("Total de cenários:", len(cenarios_exemplo))

print("\nAlterações candidatas:")
display(
    cenarios_exemplo[
        [
            "feature_alterada",
            "valor_original",
            "valor_proposto",
            "origem"
        ]
    ]
)

print("\nQuantidade de cenários por feature:")
display(
    cenarios_exemplo["feature_alterada"]
    .value_counts()
    .rename_axis("Feature")
    .reset_index(name="Cenários")
)


Campanha: KLK-0250
Total de cenários: 8

Alterações candidatas:


,feature_alterada,valor_original,valor_proposto,origem
0,has_face,0,1,Inclusão de característica ausente
1,format,vertical,horizontal,Categoria observada no treinamento
2,format,vertical,quadrado,Categoria observada no treinamento
3,text_density,medium,high,Categoria observada no treinamento
4,text_density,medium,low,Categoria observada no treinamento
5,video_duration_s,22.0,18,Quartis da duração por plataforma no treino
6,video_duration_s,22.0,26,Quartis da duração por plataforma no treino
7,video_duration_s,22.0,31,Quartis da duração por plataforma no treino



Quantidade de cenários por feature:


,Feature,Cenários
0,video_duration_s,3
1,format,2
2,text_density,2
3,has_face,1


In [6]:

def valores_iguais(a, b):
    if pd.isna(a) and pd.isna(b):
        return True

    return a == b


def validar_cenarios(campanha, cenarios):
    original = preparar_campanha(campanha).iloc[0]

    for _, cenario in cenarios.iterrows():

        # Identificar todas as features alteradas
        alteradas = [
            feature
            for feature in features_reduzidas
            if not valores_iguais(
                original[feature],
                cenario[feature]
            )
        ]

        # Exatamente uma feature deve ser modificada
        assert len(alteradas) == 1, (
            f"Cenário inválido: {alteradas}"
        )

        assert alteradas[0] == cenario["feature_alterada"]

        # Preservar o contexto
        for feature in features_contexto:
            assert valores_iguais(
                original[feature],
                cenario[feature]
            )

        # Garantir que o cenário seja aceito pelo input
        preparar_campanha(
            cenario[features_reduzidas]
        )

    print("Todos os cenários passaram na validação.")
    print("Contexto preservado e apenas uma alteração por cenário.")


validar_cenarios(
    campanha_exemplo,
    cenarios_exemplo
)


Todos os cenários passaram na validação.
Contexto preservado e apenas uma alteração por cenário.



---

## 4. Simulação preditiva com o Ridge reduzido

### 4.1. Estimativa das diferenças previstas

Após a geração dos cenários alternativos, será utilizado o Ridge reduzido desenvolvido na Parte 2 para estimar o Klike Score de cada configuração.

Para cada campanha, o procedimento será:

1. Calcular o Klike Score previsto para a configuração original.
2. Calcular o Klike Score previsto para cada cenário alternativo.
3. Obter a diferença entre a previsão alternativa e a previsão original.
4. Organizar os resultados para posterior avaliação contextual.

A diferença prevista será calculada por:

\[
\Delta_{\text{score}} = \hat{y}_{\text{alternativo}} - \hat{y}_{\text{original}}
\]

Valores positivos representam cenários com maior Klike Score previsto pelo modelo, enquanto valores negativos representam cenários com previsão inferior.

### Limitações

As diferenças estimadas representam comparações entre previsões de um modelo estatístico, não efeitos causais comprovados.

Além disso, o Ridge é um modelo linear aditivo, sem interações explícitas entre características criativas e plataformas. Portanto, diferenças específicas por contexto serão investigadas separadamente na análise estatística.

O modelo não será ajustado novamente nesta etapa. Serão utilizados somente os artefatos previamente treinados.

As previsões serão mantidas em sua escala original, sem truncamento automático, para preservar as diferenças estimadas e permitir identificar eventuais valores fora da faixa de 0 a 100.

A seleção definitiva das recomendações dependerá também da consistência das associações observadas no conjunto de treinamento.


In [9]:

def simular_impactos(campanha):
    """
    Calcula diferenças previstas no Klike Score
    entre uma campanha e seus cenários alternativos.

    Utiliza o Ridge reduzido já treinado,
    sem realizar novos ajustes.
    """

    # Preparar campanha original
    entrada_original = preparar_campanha(campanha)

    # Gerar alternativas válidas
    cenarios = gerar_cenarios(campanha)

    # Previsão da campanha original
    score_original = float(
        modelo_ridge.predict(entrada_original)[0]
    )

    colunas_resultado = [
        "feature_alterada",
        "valor_original",
        "valor_proposto",
        "origem",
        "score_previsto_original",
        "score_previsto_cenario",
        "delta_score_previsto"
    ]

    # Tratar campanhas sem alterações candidatas
    if cenarios.empty:
        return score_original, pd.DataFrame(
            columns=colunas_resultado
        )

    # Preparar alternativas para o mesmo pipeline
    X_cenarios = cenarios[features_reduzidas].copy()

    # Previsões sem novo treinamento
    scores_alternativos = modelo_ridge.predict(
        X_cenarios
    )

    # Organizar os resultados
    resultados = cenarios[
        [
            "feature_alterada",
            "valor_original",
            "valor_proposto",
            "origem"
        ]
    ].copy()

    resultados["score_previsto_original"] = score_original
    resultados["score_previsto_cenario"] = scores_alternativos

    resultados["delta_score_previsto"] = (
        resultados["score_previsto_cenario"]
        - resultados["score_previsto_original"]
    )

    # Ordenação exploratória pelo delta previsto
    resultados = resultados.sort_values(
        by="delta_score_previsto",
        ascending=False,
        kind="stable"
    ).reset_index(drop=True)

    return score_original, resultados


In [10]:

# Simular a campanha utilizada nas etapas anteriores
score_original, simulacoes = simular_impactos(
    campanha_exemplo
)

print("Campanha:", campanha_exemplo["campaign_id"])
print(f"Klike Score previsto original: {score_original:.3f}")

print("\nQuantidade de cenários avaliados:", len(simulacoes))

print("\nResultados da simulação preditiva:")

display(
    simulacoes.round({
        "score_previsto_original": 3,
        "score_previsto_cenario": 3,
        "delta_score_previsto": 3
    })
)

# Verificações de consistência
assert np.isfinite(score_original)

assert np.isfinite(
    simulacoes[
        [
            "score_previsto_original",
            "score_previsto_cenario",
            "delta_score_previsto"
        ]
    ].to_numpy(dtype=float)
).all()

assert np.allclose(
    simulacoes["delta_score_previsto"],
    simulacoes["score_previsto_cenario"]
    - simulacoes["score_previsto_original"]
)

# Identificar eventuais previsões fora da escala
previsoes_fora_faixa = (
    ~simulacoes["score_previsto_cenario"].between(0, 100)
).sum()

print("\nVerificações:")
print("Diferenças calculadas corretamente.")
print("Previsões alternativas fora de 0 a 100:", previsoes_fora_faixa)
print(
    "Previsão original fora de 0 a 100:",
    not 0 <= score_original <= 100
)


Campanha: KLK-0250
Klike Score previsto original: 66.506

Quantidade de cenários avaliados: 8

Resultados da simulação preditiva:


,feature_alterada,valor_original,valor_proposto,origem,score_previsto_original,score_previsto_cenario,delta_score_previsto
0,has_face,0,1,Inclusão de característica ausente,66.506,78.256,11.750
1,video_duration_s,22.0,31,Quartis da duração por plataforma no treino,66.506,66.584,0.078
2,video_duration_s,22.0,26,Quartis da duração por plataforma no treino,66.506,66.541,0.035
3,video_duration_s,22.0,18,Quartis da duração por plataforma no treino,66.506,66.471,-0.035
4,format,vertical,quadrado,Categoria observada no treinamento,66.506,65.179,-1.326
5,format,vertical,horizontal,Categoria observada no treinamento,66.506,64.969,-1.537
6,text_density,medium,low,Categoria observada no treinamento,66.506,59.922,-6.584
7,text_density,medium,high,Categoria observada no treinamento,66.506,52.471,-14.035



Verificações:
Diferenças calculadas corretamente.
Previsões alternativas fora de 0 a 100: 0
Previsão original fora de 0 a 100: False



---

## 5. Evidências estatísticas contextuais

### 5.1. Preparação da base de referência

O Recommendations Engine híbrido combinará duas fontes complementares de informação:

- **Previsões do Ridge reduzido:** diferenças estimadas no Klike Score entre configurações alternativas.
- **Associações observacionais:** diferenças de desempenho entre grupos de campanhas que compartilham características de contexto.

As análises contextuais utilizarão exclusivamente os 400 registros de treinamento, evitando utilizar o conjunto de teste na construção das recomendações.

Serão consideradas inicialmente três métricas:

- `klike_score`: pontuação de qualidade do criativo.
- `ctr`: proporção de impressões que resultaram em cliques.
- `engagement_rate`: proporção de interações em relação às impressões.

Na Parte 1, foram identificadas inconsistências matemáticas no CTR. Portanto, será reproduzida a regra de tratamento utilizada na EDA: divergências superiores a 0,0001 entre o CTR registrado e o CTR recalculado serão desconsideradas nas análises dessa métrica.

Essa exclusão será aplicada apenas ao CTR, preservando as demais informações da campanha.

As comparações serão descritivas e não constituirão estimativas de efeitos causais.


In [11]:

# Base utilizada exclusivamente para evidências contextuais
df_evidencias = df_train.copy()

# Métricas de interesse
metricas_evidencia = [
    "klike_score",
    "ctr",
    "engagement_rate"
]

# Reproduzir a verificação de consistência do CTR da Parte 1
ctr_calculado = (
    df_evidencias["clicks"]
    / df_evidencias["impressions"].replace(0, np.nan)
)

tolerancia_ctr = 0.0001

ctr_comparavel = (
    df_evidencias["ctr"].notna()
    & ctr_calculado.notna()
)

ctr_inconsistente = (
    ctr_comparavel
    & (
        df_evidencias["ctr"] - ctr_calculado
    ).abs().gt(tolerancia_ctr)
)

# Preservar o valor original e criar versão tratada
df_evidencias["ctr_consistente"] = (
    df_evidencias["ctr"].mask(ctr_inconsistente)
)

# Resumo da disponibilidade das métricas
resumo_metricas = pd.DataFrame([
    {
        "Métrica": "Klike Score",
        "Registros disponíveis": df_evidencias[
            "klike_score"
        ].notna().sum()
    },
    {
        "Métrica": "CTR consistente",
        "Registros disponíveis": df_evidencias[
            "ctr_consistente"
        ].notna().sum()
    },
    {
        "Métrica": "Engagement Rate",
        "Registros disponíveis": df_evidencias[
            "engagement_rate"
        ].notna().sum()
    }
])

print("Preparação das evidências contextuais")
print(f"Registros no treinamento: {len(df_evidencias)}")
print(f"CTR comparáveis: {ctr_comparavel.sum()}")
print(f"CTR inconsistentes: {ctr_inconsistente.sum()}")

print("\nDisponibilidade das métricas:")
display(resumo_metricas)

# Verificações de isolamento dos dados
assert len(df_evidencias) == len(X_train)

assert set(df_evidencias.index).isdisjoint(
    set(X_test.index)
)

assert df_evidencias["ctr"].equals(
    df_train["ctr"]
)

print("\nVerificações de isolamento concluídas.")


Preparação das evidências contextuais
Registros no treinamento: 400
CTR comparáveis: 400
CTR inconsistentes: 49

Disponibilidade das métricas:


,Métrica,Registros disponíveis
0,Klike Score,400
1,CTR consistente,351
2,Engagement Rate,375



Verificações de isolamento concluídas.



### 5.2. Distribuição das campanhas por contexto

Antes de calcular associações, será avaliada a quantidade de campanhas disponíveis em cada combinação de plataforma e categoria.

Esse diagnóstico é importante porque a segmentação excessiva pode produzir comparações baseadas em poucas observações, aumentando a instabilidade das estimativas.

A arquitetura proposta seguirá uma hierarquia de contextualização:

1. **Plataforma e categoria:** comparação mais específica, quando houver suporte amostral suficiente nos dois grupos.
2. **Plataforma:** comparação menos específica, utilizada quando a combinação anterior não possuir observações suficientes.
3. **Evidência insuficiente:** não será apresentada uma associação quantitativa contextual quando nenhum dos níveis atender aos critérios mínimos.

A primeira referência para suporte amostral será de pelo menos 10 observações válidas por grupo, seguindo o critério exploratório utilizado na Parte 1.

Essa regra é uma salvaguarda pragmática, não uma garantia de precisão estatística. Outras limitações, como variabilidade, confundimento e múltiplas comparações, também deverão ser consideradas.


In [12]:

print("Distribuição por plataforma:")

display(
    df_evidencias["platform"]
    .value_counts()
    .rename_axis("Plataforma")
    .reset_index(name="Campanhas")
)

print("\nDistribuição por plataforma e categoria:")

distribuicao_contexto = pd.crosstab(
    df_evidencias["platform"],
    df_evidencias["category"]
)

display(distribuicao_contexto)

print("\nCampanhas disponíveis no contexto do exemplo:")

contexto_exemplo = df_evidencias.loc[
    (df_evidencias["platform"] == campanha_exemplo["platform"])
    & (df_evidencias["category"] == campanha_exemplo["category"])
]

print(
    f"Plataforma: {campanha_exemplo['platform']}"
)
print(
    f"Categoria: {campanha_exemplo['category']}"
)
print(
    f"Campanhas no contexto: {len(contexto_exemplo)}"
)


Distribuição por plataforma:


,Plataforma,Campanhas
0,Meta,188
1,TikTok,142
2,LinkedIn,70



Distribuição por plataforma e categoria:


category,App Install,Branding,E-commerce,Lead Gen,SaaS
platform,,,,,
LinkedIn,15,8,24,12,11
Meta,32,22,61,36,37
TikTok,20,17,46,32,27



Campanhas disponíveis no contexto do exemplo:
Plataforma: LinkedIn
Categoria: App Install
Campanhas no contexto: 15



### 5.3. Comparação de desempenho entre configurações criativas

Para complementar as simulações do Ridge, serão calculadas diferenças observacionais entre campanhas com características criativas distintas.

Cada comparação considerará a configuração atual e uma configuração alternativa, utilizando uma hierarquia de contexto:

1. **Plataforma + categoria:** prioridade para campanhas do mesmo ambiente e segmento.
2. **Plataforma:** alternativa quando o agrupamento anterior possuir dados insuficientes.
3. **Evidência insuficiente:** resultado apresentado quando não houver suporte amostral adequado.

Será exigido um mínimo inicial de 10 observações válidas por grupo, para cada métrica analisada.

As comparações utilizarão médias dos grupos, apresentando:

- Quantidade de observações em cada grupo.
- Média observada na configuração original.
- Média observada na configuração alternativa.
- Diferença absoluta entre as médias.
- Diferença percentual relativa, quando aplicável.

Para o Klike Score, a diferença absoluta será expressa em pontos. Para CTR e taxa de engajamento, será expressa em pontos percentuais, acompanhada da variação percentual relativa.

**Importante:** diferenças observadas entre campanhas não representam o impacto causal de modificar um criativo. Os grupos podem diferir em outras características relevantes.

As comparações serão realizadas exclusivamente com os dados de treinamento. As variáveis numéricas, como a duração do vídeo, serão tratadas separadamente, pois exigem uma estratégia de agrupamento apropriada.


In [13]:

def comparar_cenario_contextual(
    campanha,
    feature,
    valor_proposto,
    minimo_por_grupo=10
):
    """
    Compara métricas observadas entre duas configurações
    de uma característica criativa.

    Utiliza somente df_evidencias (treinamento).

    A função não estima efeitos causais.
    """

    entrada = preparar_campanha(campanha).iloc[0]

    features_comparaveis = [
        "has_subtitle",
        "has_cta",
        "has_hook",
        "has_face",
        "format",
        "text_density"
    ]

    if feature not in features_comparaveis:
        raise ValueError(
            f"Feature não suportada nesta comparação: {feature}"
        )

    valor_original = entrada[feature]

    if pd.isna(valor_original):
        raise ValueError(
            "Não é possível comparar uma feature de valor desconhecido."
        )

    if feature in features_booleanas:
        if valor_proposto not in (0, 1, True, False):
            raise ValueError("Valor booleano proposto inválido.")

        valor_proposto = int(valor_proposto)

    elif valor_proposto not in categorias_validas[feature]:
        raise ValueError("Categoria proposta não observada no treino.")

    if valor_original == valor_proposto:
        raise ValueError(
            "A configuração proposta deve diferir da original."
        )

    metricas = {
        "klike_score": "Klike Score",
        "ctr_consistente": "CTR consistente",
        "engagement_rate": "Engagement Rate"
    }

    # Hierarquia de contextos, sem utilizar o conjunto de teste
    contextos = [
        (
            "Plataforma + categoria",
            df_evidencias.loc[
                (df_evidencias["platform"] == entrada["platform"])
                & (df_evidencias["category"] == entrada["category"])
            ]
        ),
        (
            "Plataforma",
            df_evidencias.loc[
                df_evidencias["platform"] == entrada["platform"]
            ]
        )
    ]

    resultados = []

    for coluna_metrica, nome_metrica in metricas.items():

        comparacao = None
        ultimo_suporte = (0, 0)

        for nivel_contexto, dados_contexto in contextos:

            dados_validos = dados_contexto.loc[
                dados_contexto[coluna_metrica].notna()
                & dados_contexto[feature].notna()
            ]

            grupo_original = dados_validos.loc[
                dados_validos[feature] == valor_original,
                coluna_metrica
            ]

            grupo_proposto = dados_validos.loc[
                dados_validos[feature] == valor_proposto,
                coluna_metrica
            ]

            n_original = len(grupo_original)
            n_proposto = len(grupo_proposto)

            ultimo_suporte = (n_original, n_proposto)

            if (
                n_original < minimo_por_grupo
                or n_proposto < minimo_por_grupo
            ):
                continue

            media_original = grupo_original.mean()
            media_proposta = grupo_proposto.mean()

            diferenca = media_proposta - media_original

            if coluna_metrica == "klike_score":
                unidade = "pontos"
                media_exibida_original = media_original
                media_exibida_proposta = media_proposta
                diferenca_exibida = diferenca
                variacao_relativa = np.nan

            else:
                unidade = "p.p."
                media_exibida_original = media_original * 100
                media_exibida_proposta = media_proposta * 100
                diferenca_exibida = diferenca * 100

                variacao_relativa = (
                    (diferenca / media_original) * 100
                    if media_original != 0
                    else np.nan
                )

            comparacao = {
                "Métrica": nome_metrica,
                "Contexto": nivel_contexto,
                "N original": n_original,
                "N proposto": n_proposto,
                "Média original": media_exibida_original,
                "Média proposta": media_exibida_proposta,
                "Diferença observada": diferenca_exibida,
                "Unidade": unidade,
                "Variação relativa (%)": variacao_relativa,
                "Status": "Comparação disponível"
            }

            break

        if comparacao is None:
            comparacao = {
                "Métrica": nome_metrica,
                "Contexto": "Plataforma",
                "N original": ultimo_suporte[0],
                "N proposto": ultimo_suporte[1],
                "Média original": np.nan,
                "Média proposta": np.nan,
                "Diferença observada": np.nan,
                "Unidade": (
                    "pontos"
                    if coluna_metrica == "klike_score"
                    else "p.p."
                ),
                "Variação relativa (%)": np.nan,
                "Status": "Amostra insuficiente"
            }

        resultados.append(comparacao)

    return pd.DataFrame(resultados)


In [14]:

# Comparação contextual da principal alteração candidata
evidencias_rosto = comparar_cenario_contextual(
    campanha=campanha_exemplo,
    feature="has_face",
    valor_proposto=1,
    minimo_por_grupo=10
)

print("Campanha:", campanha_exemplo["campaign_id"])
print("Plataforma:", campanha_exemplo["platform"])
print("Categoria:", campanha_exemplo["category"])

print("\nAlteração avaliada:")
print("Adicionar rosto humano ao criativo")

print("\nEvidências estatísticas contextuais:")

display(
    evidencias_rosto.round({
        "Média original": 3,
        "Média proposta": 3,
        "Diferença observada": 3,
        "Variação relativa (%)": 2
    })
)

print("\nVerificação do suporte amostral:")

for _, resultado in evidencias_rosto.iterrows():

    print(
        f"{resultado['Métrica']}: "
        f"{resultado['Status']} | "
        f"{resultado['Contexto']} | "
        f"N={resultado['N original']} vs "
        f"N={resultado['N proposto']}"
    )


Campanha: KLK-0250
Plataforma: LinkedIn
Categoria: App Install

Alteração avaliada:
Adicionar rosto humano ao criativo

Evidências estatísticas contextuais:


,Métrica,Contexto,N original,N proposto,Média original,Média proposta,Diferença observada,Unidade,Variação relativa (%),Status
0,Klike Score,Plataforma,30,40,48.733,62.295,13.562,pontos,NaN,Comparação disponível
1,CTR consistente,Plataforma,27,35,6.057,8.626,2.570,p.p.,42.43,Comparação disponível
2,Engagement Rate,Plataforma,29,37,1.822,2.457,0.635,p.p.,34.87,Comparação disponível



Verificação do suporte amostral:
Klike Score: Comparação disponível | Plataforma | N=30 vs N=40
CTR consistente: Comparação disponível | Plataforma | N=27 vs N=35
Engagement Rate: Comparação disponível | Plataforma | N=29 vs N=37



### 5.4. Integração entre previsões e evidências contextuais

Nesta etapa, as simulações preditivas serão associadas às comparações estatísticas contextuais.

Para cada cenário categórico, serão reunidas:

- A característica modificada e sua configuração alternativa.
- A diferença prevista pelo Ridge reduzido.
- A diferença média observada no Klike Score.
- As diferenças observadas no CTR e na taxa de engajamento.
- O contexto e o suporte amostral de cada comparação.

Os resultados serão organizados em duas estruturas:

1. **Tabela consolidada:** uma linha por cenário, contendo as diferenças preditivas e observacionais disponíveis.
2. **Tabela detalhada:** uma linha por cenário e métrica, preservando o contexto, os tamanhos amostrais e o status da comparação.

As alterações de duração serão mantidas na tabela consolidada, mas suas evidências contextuais permanecerão pendentes até a definição de uma metodologia apropriada para variáveis numéricas.

Nesta etapa, não serão atribuídos pesos às métricas nem combinadas suas diferenças em um único indicador.

A integração permitirá investigar posteriormente a concordância entre o Ridge e os padrões observados, sem confundir previsão com causalidade.


In [16]:

def integrar_evidencias_cenarios(
    campanha,
    simulacoes,
    minimo_por_grupo=10
):
    """
    Integra previsões do Ridge e evidências contextuais.

    Retorna:
    - resumo: uma linha por cenário.
    - detalhes: uma linha por cenário e métrica avaliada.

    Não classifica causalidade nem gera recomendações.
    """

    colunas_metricas = {
        "Klike Score": "Delta Klike observado (pontos)",
        "CTR consistente": "Delta CTR observado (p.p.)",
        "Engagement Rate": "Delta Engagement observado (p.p.)"
    }

    resumos = []
    detalhes = []

    for id_cenario, (_, cenario) in enumerate(
        simulacoes.iterrows(), start=1
    ):
        feature = cenario["feature_alterada"]

        resumo = {
            "Cenario": id_cenario,
            "Feature": feature,
            "Original": cenario["valor_original"],
            "Proposto": cenario["valor_proposto"],
            "Delta Ridge (pontos)": float(
                cenario["delta_score_previsto"]
            ),
            "Status evidencias": None
        }

        for coluna in colunas_metricas.values():
            resumo[coluna] = np.nan

        # A duração requer um método contextual específico
        if feature == "video_duration_s":
            resumo["Status evidencias"] = (
                "Pendente: duracao numerica"
            )
            resumos.append(resumo)
            continue

        evidencias = comparar_cenario_contextual(
            campanha=campanha,
            feature=feature,
            valor_proposto=cenario["valor_proposto"],
            minimo_por_grupo=minimo_por_grupo
        )

        n_disponiveis = (
            evidencias["Status"] == "Comparação disponível"
        ).sum()

        if n_disponiveis == len(colunas_metricas):
            resumo["Status evidencias"] = "Completa"
        elif n_disponiveis > 0:
            resumo["Status evidencias"] = "Parcial"
        else:
            resumo["Status evidencias"] = "Insuficiente"

        for _, evidencia in evidencias.iterrows():
            metrica = evidencia["Métrica"]
            disponivel = (
                evidencia["Status"] == "Comparação disponível"
            )

            if disponivel:
                resumo[colunas_metricas[metrica]] = float(
                    evidencia["Diferença observada"]
                )

            detalhes.append({
                "Cenario": id_cenario,
                "Feature": feature,
                "Proposto": cenario["valor_proposto"],
                "Metrica": metrica,
                "Contexto": evidencia["Contexto"],
                "N original": evidencia["N original"],
                "N proposto": evidencia["N proposto"],
                "Diferenca observada": evidencia[
                    "Diferença observada"
                ],
                "Unidade": evidencia["Unidade"],
                "Variacao relativa (%)": evidencia[
                    "Variação relativa (%)"
                ],
                "Status": evidencia["Status"]
            })

        resumos.append(resumo)

    return pd.DataFrame(resumos), pd.DataFrame(detalhes)


In [17]:

# Integrar os cenários da campanha KLK-0250
resumo_integrado, detalhes_integrados = (
    integrar_evidencias_cenarios(
        campanha=campanha_exemplo,
        simulacoes=simulacoes,
        minimo_por_grupo=10
    )
)

print("Campanha:", campanha_exemplo["campaign_id"])
print("Total de cenários:", len(resumo_integrado))

print("\nResumo integrado:")
display(resumo_integrado.round(3))

print("\nDisponibilidade das evidências:")
display(
    resumo_integrado["Status evidencias"]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Cenários")
)

# Exibir os detalhes da alteração has_face
cenario_rosto = resumo_integrado.loc[
    resumo_integrado["Feature"] == "has_face",
    "Cenario"
].iloc[0]

print("\nEvidências detalhadas — has_face:")
display(
    detalhes_integrados.loc[
        detalhes_integrados["Cenario"] == cenario_rosto
    ].round(3)
)

# Verificações de consistência
assert len(resumo_integrado) == len(simulacoes)

n_cenarios_categoricos = (
    simulacoes["feature_alterada"] != "video_duration_s"
).sum()

assert len(detalhes_integrados) == (
    n_cenarios_categoricos * 3
)

assert resumo_integrado["Cenario"].is_unique

assert np.allclose(
    resumo_integrado["Delta Ridge (pontos)"],
    simulacoes["delta_score_previsto"]
)

print("\nIntegração concluída com sucesso.")


Campanha: KLK-0250
Total de cenários: 8

Resumo integrado:


,Cenario,Feature,Original,Proposto,Delta Ridge (pontos),Status evidencias,Delta Klike observado (pontos),Delta CTR observado (p.p.),Delta Engagement observado (p.p.)
0,1,has_face,0,1,11.750,Completa,13.562,2.570,0.635
1,2,video_duration_s,22.0,31,0.078,Pendente: duracao numerica,NaN,NaN,NaN
2,3,video_duration_s,22.0,26,0.035,Pendente: duracao numerica,NaN,NaN,NaN
3,4,video_duration_s,22.0,18,-0.035,Pendente: duracao numerica,NaN,NaN,NaN
4,5,format,vertical,quadrado,-1.326,Parcial,14.089,NaN,-0.151
5,6,format,vertical,horizontal,-1.537,Parcial,16.459,NaN,-0.140
6,7,text_density,medium,low,-6.584,Completa,-5.090,0.909,-0.354
7,8,text_density,medium,high,-14.035,Completa,-15.875,-0.563,-0.162



Disponibilidade das evidências:


,Status,Cenários
0,Completa,3
1,Pendente: duracao numerica,3
2,Parcial,2



Evidências detalhadas — has_face:


,Cenario,Feature,Proposto,Metrica,Contexto,N original,N proposto,Diferenca observada,Unidade,Variacao relativa (%),Status
0,1,has_face,1,Klike Score,Plataforma,30,40,13.562,pontos,NaN,Comparação disponível
1,1,has_face,1,CTR consistente,Plataforma,27,35,2.570,p.p.,42.426,Comparação disponível
2,1,has_face,1,Engagement Rate,Plataforma,29,37,0.635,p.p.,34.874,Comparação disponível



Integração concluída com sucesso.



### 5.5. Análise contextual da duração do vídeo

A duração do vídeo é uma variável numérica contínua. Diferentemente das características booleanas e categóricas, comparações entre valores exatos podem produzir grupos pequenos e estimativas instáveis.

Para investigar associações entre duração e desempenho, os vídeos serão organizados em quatro faixas, definidas pelos quartis observados no conjunto de treinamento de cada plataforma.

Essa estratégia permite comparar campanhas de duração semelhante sem estabelecer arbitrariamente um tempo ideal.

Para cada cenário:

1. Identificar a faixa de duração da campanha original.
2. Identificar a faixa correspondente à duração proposta.
3. Comparar as métricas observadas entre essas faixas.
4. Priorizar o contexto plataforma + categoria quando houver pelo menos 10 observações válidas por grupo.
5. Utilizar o contexto da plataforma quando o grupo mais específico for insuficiente.

Quando as durações original e proposta pertencerem à mesma faixa, não será atribuída uma diferença observacional à alteração.

Os limites das faixas serão definidos exclusivamente com os dados de treinamento, evitando data leakage.

**Limitação:** os resultados representarão diferenças entre faixas de duração, não o impacto causal ou exato de modificar um vídeo de uma duração específica para outra.


In [18]:

def comparar_duracao_contextual(
    campanha,
    duracao_proposta,
    minimo_por_grupo=10
):
    """
    Compara faixas de duração utilizando os quartis
    da plataforma calculados somente no treinamento.

    As diferenças são observacionais, não causais.
    """

    entrada = preparar_campanha(campanha).iloc[0]
    duracao_original = entrada["video_duration_s"]

    if pd.isna(duracao_original):
        raise ValueError(
            "A duração original é desconhecida."
        )

    # Validar a duração alternativa
    alternativa = entrada.to_dict()
    alternativa["video_duration_s"] = duracao_proposta

    duracao_proposta = preparar_campanha(
        alternativa
    ).iloc[0]["video_duration_s"]

    if pd.isna(duracao_proposta):
        raise ValueError(
            "A duração proposta não pode ser nula."
        )

    if np.isclose(duracao_original, duracao_proposta):
        raise ValueError(
            "A duração proposta deve ser diferente da original."
        )

    # Quartis calculados apenas com dados de treinamento
    dados_plataforma = df_evidencias.loc[
        df_evidencias["platform"] == entrada["platform"]
    ]

    cortes = dados_plataforma[
        "video_duration_s"
    ].dropna().quantile([0.25, 0.50, 0.75]).to_numpy()

    if not np.all(np.isfinite(cortes)) or not np.all(
        np.diff(cortes) > 0
    ):
        raise ValueError(
            "Não foi possível construir quatro faixas distintas."
        )

    def identificar_faixa(duracao):
        return int(
            np.searchsorted(cortes, duracao, side="left") + 1
        )

    faixa_original = identificar_faixa(duracao_original)
    faixa_proposta = identificar_faixa(duracao_proposta)

    metricas = {
        "klike_score": "Klike Score",
        "ctr_consistente": "CTR consistente",
        "engagement_rate": "Engagement Rate"
    }

    contextos = [
        (
            "Plataforma + categoria",
            dados_plataforma.loc[
                dados_plataforma["category"] == entrada["category"]
            ]
        ),
        (
            "Plataforma",
            dados_plataforma
        )
    ]

    resultados = []

    for coluna, nome_metrica in metricas.items():

        unidade = (
            "pontos"
            if coluna == "klike_score"
            else "p.p."
        )

        resultado = {
            "Métrica": nome_metrica,
            "Contexto": None,
            "Faixa original": faixa_original,
            "Faixa proposta": faixa_proposta,
            "N original": 0,
            "N proposto": 0,
            "Diferença observada": np.nan,
            "Unidade": unidade,
            "Variação relativa (%)": np.nan,
            "Status": "Amostra insuficiente"
        }

        if faixa_original == faixa_proposta:
            resultado["Status"] = "Mesma faixa de duração"
            resultados.append(resultado)
            continue

        for nivel, dados_contexto in contextos:

            dados_validos = dados_contexto.dropna(
                subset=["video_duration_s", coluna]
            )

            faixas = np.searchsorted(
                cortes,
                dados_validos["video_duration_s"].to_numpy(),
                side="left"
            ) + 1

            grupo_original = dados_validos.loc[
                faixas == faixa_original, coluna
            ]

            grupo_proposto = dados_validos.loc[
                faixas == faixa_proposta, coluna
            ]

            n_original = len(grupo_original)
            n_proposto = len(grupo_proposto)

            resultado.update({
                "Contexto": nivel,
                "N original": n_original,
                "N proposto": n_proposto
            })

            if (
                n_original < minimo_por_grupo
                or n_proposto < minimo_por_grupo
            ):
                continue

            media_original = grupo_original.mean()
            media_proposta = grupo_proposto.mean()

            diferenca = media_proposta - media_original

            if coluna != "klike_score":
                diferenca *= 100

            variacao_relativa = (
                100 * (
                    media_proposta - media_original
                ) / media_original
                if coluna != "klike_score"
                and media_original != 0
                else np.nan
            )

            resultado.update({
                "Diferença observada": diferenca,
                "Variação relativa (%)": variacao_relativa,
                "Status": "Comparação disponível"
            })

            break

        resultados.append(resultado)

    return pd.DataFrame(resultados)


In [19]:

# Cenários de duração da campanha de exemplo
cenarios_duracao = simulacoes.loc[
    simulacoes["feature_alterada"] == "video_duration_s"
].copy()

# Identificar os quartis de duração da plataforma
duracoes_referencia = df_evidencias.loc[
    df_evidencias["platform"] == campanha_exemplo["platform"],
    "video_duration_s"
].dropna()

print("Campanha:", campanha_exemplo["campaign_id"])
print("Plataforma:", campanha_exemplo["platform"])

print("\nQuartis da duração no treinamento:")
display(
    duracoes_referencia.quantile(
        [0.25, 0.50, 0.75]
    ).to_frame(name="Duração (s)")
)

# Comparar cada duração alternativa
resultados_duracao = []

for _, cenario in cenarios_duracao.iterrows():

    comparacao = comparar_duracao_contextual(
        campanha=campanha_exemplo,
        duracao_proposta=cenario["valor_proposto"],
        minimo_por_grupo=10
    )

    comparacao.insert(
        0,
        "Duração proposta (s)",
        float(cenario["valor_proposto"])
    )

    comparacao.insert(
        1,
        "Delta Ridge (pontos)",
        float(cenario["delta_score_previsto"])
    )

    resultados_duracao.append(comparacao)

evidencias_duracao = pd.concat(
    resultados_duracao,
    ignore_index=True
)

print("\nComparações contextuais da duração:")
display(evidencias_duracao.round(3))

print("\nStatus das comparações:")
display(
    evidencias_duracao["Status"]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Comparações")
)

# Verificações
assert len(evidencias_duracao) == (
    len(cenarios_duracao) * 3
)

assert set(evidencias_duracao["Métrica"]) == {
    "Klike Score",
    "CTR consistente",
    "Engagement Rate"
}

print("\nAnálise contextual de duração concluída.")


Campanha: KLK-0250
Plataforma: LinkedIn

Quartis da duração no treinamento:


,Duração (s)
0.25,18.50
0.50,25.50
0.75,31.25



Comparações contextuais da duração:


,Duração proposta (s),Delta Ridge (pontos),Métrica,Contexto,Faixa original,Faixa proposta,N original,N proposto,Diferença observada,Unidade,Variação relativa (%),Status
0,31.0,0.078,Klike Score,Plataforma,2,3,16,16,4.600,pontos,NaN,Comparação disponível
1,31.0,0.078,CTR consistente,Plataforma,2,3,14,14,1.893,p.p.,25.161,Comparação disponível
2,31.0,0.078,Engagement Rate,Plataforma,2,3,14,16,0.235,p.p.,10.108,Comparação disponível
3,26.0,0.035,Klike Score,Plataforma,2,3,16,16,4.600,pontos,NaN,Comparação disponível
4,26.0,0.035,CTR consistente,Plataforma,2,3,14,14,1.893,p.p.,25.161,Comparação disponível
5,26.0,0.035,Engagement Rate,Plataforma,2,3,14,16,0.235,p.p.,10.108,Comparação disponível
6,18.0,-0.035,Klike Score,Plataforma,2,1,16,16,-1.688,pontos,NaN,Comparação disponível
7,18.0,-0.035,CTR consistente,Plataforma,2,1,14,16,-1.193,p.p.,-15.856,Comparação disponível
8,18.0,-0.035,Engagement Rate,Plataforma,2,1,14,14,-0.306,p.p.,-13.149,Comparação disponível



Status das comparações:


,Status,Comparações
0,Comparação disponível,9



Análise contextual de duração concluída.



### 5.6. Avaliação da robustez das associações observadas

Diferenças entre médias podem ser influenciadas pela variabilidade dos dados, especialmente quando os grupos possuem poucas observações.

Para investigar a estabilidade das associações contextuais, será utilizado o método de reamostragem **bootstrap não paramétrico**, com 2.000 reamostragens independentes dentro de cada grupo.

Em cada repetição, serão calculadas as médias dos grupos original e alternativo, seguidas de sua diferença.

A distribuição das diferenças permitirá construir intervalos percentílicos de 95%.

Inicialmente, serão avaliadas:

- Inclusão de rosto humano, principal alteração com previsão positiva no Ridge.
- Aumento da duração de 22 para 31 segundos.
- Redução da duração de 22 para 18 segundos.

A alternativa de 26 segundos não será repetida nesta análise porque pertence à mesma faixa de duração da alternativa de 31 segundos, produzindo uma comparação observacional idêntica.

Serão utilizadas as três métricas: Klike Score, CTR consistente e Engagement Rate.

**Limitações:** os intervalos quantificam a variabilidade amostral das diferenças observadas, condicionadas aos grupos e contextos selecionados. Eles não controlam confundimento, viés de seleção ou múltiplas comparações e não devem ser interpretados como evidência de causalidade.

Esta análise permanecerá exploratória e utilizará exclusivamente os registros de treinamento.


In [20]:

def bootstrap_contextual(
    campanha,
    feature,
    valor_proposto,
    metrica="klike_score",
    n_bootstrap=2000,
    random_state=42
):
    """
    Calcula intervalo percentílico bootstrap para
    a diferença observacional entre dois grupos.

    Preserva os contextos e as regras de comparação
    definidos nas funções anteriores.
    """

    nomes_metricas = {
        "klike_score": "Klike Score",
        "ctr_consistente": "CTR consistente",
        "engagement_rate": "Engagement Rate"
    }

    if metrica not in nomes_metricas:
        raise ValueError("Métrica não suportada.")

    entrada = preparar_campanha(campanha).iloc[0]

    # Recuperar a comparação previamente definida
    if feature == "video_duration_s":
        comparacao = comparar_duracao_contextual(
            campanha,
            valor_proposto
        )
    else:
        comparacao = comparar_cenario_contextual(
            campanha,
            feature,
            valor_proposto
        )

    info = comparacao.loc[
        comparacao["Métrica"] == nomes_metricas[metrica]
    ].iloc[0]

    if info["Status"] != "Comparação disponível":
        return {
            "Feature": feature,
            "Proposto": valor_proposto,
            "Métrica": nomes_metricas[metrica],
            "Status": info["Status"]
        }

    # Recuperar o contexto utilizado
    dados = df_evidencias.loc[
        df_evidencias["platform"] == entrada["platform"]
    ].copy()

    if info["Contexto"] == "Plataforma + categoria":
        dados = dados.loc[
            dados["category"] == entrada["category"]
        ].copy()

    dados = dados.dropna(subset=[feature, metrica])

    # Identificar os grupos de comparação
    if feature == "video_duration_s":

        referencia = df_evidencias.loc[
            df_evidencias["platform"] == entrada["platform"],
            "video_duration_s"
        ].dropna()

        cortes = referencia.quantile(
            [0.25, 0.50, 0.75]
        ).to_numpy()

        faixas = np.searchsorted(
            cortes,
            dados["video_duration_s"].to_numpy(),
            side="left"
        ) + 1

        grupo_original = dados.loc[
            faixas == info["Faixa original"],
            metrica
        ].to_numpy(dtype=float)

        grupo_proposto = dados.loc[
            faixas == info["Faixa proposta"],
            metrica
        ].to_numpy(dtype=float)

    else:

        grupo_original = dados.loc[
            dados[feature] == entrada[feature],
            metrica
        ].to_numpy(dtype=float)

        grupo_proposto = dados.loc[
            dados[feature] == valor_proposto,
            metrica
        ].to_numpy(dtype=float)

    # Verificar consistência com os grupos anteriores
    assert len(grupo_original) == info["N original"]
    assert len(grupo_proposto) == info["N proposto"]

    rng = np.random.default_rng(random_state)

    # Reamostragem independente dentro de cada grupo
    medias_originais = rng.choice(
        grupo_original,
        size=(n_bootstrap, len(grupo_original)),
        replace=True
    ).mean(axis=1)

    medias_propostas = rng.choice(
        grupo_proposto,
        size=(n_bootstrap, len(grupo_proposto)),
        replace=True
    ).mean(axis=1)

    diferencas = medias_propostas - medias_originais

    limite_inferior, limite_superior = np.quantile(
        diferencas,
        [0.025, 0.975]
    )

    diferenca_observada = (
        grupo_proposto.mean() - grupo_original.mean()
    )

    # Converter taxas em pontos percentuais
    fator = 1 if metrica == "klike_score" else 100

    return {
        "Feature": feature,
        "Proposto": valor_proposto,
        "Métrica": nomes_metricas[metrica],
        "Contexto": info["Contexto"],
        "N original": len(grupo_original),
        "N proposto": len(grupo_proposto),
        "Diferença observada": diferenca_observada * fator,
        "IC 95% inferior": limite_inferior * fator,
        "IC 95% superior": limite_superior * fator,
        "Inclui zero": limite_inferior <= 0 <= limite_superior,
        "Unidade": "pontos" if fator == 1 else "p.p.",
        "Status": "Bootstrap concluído"
    }


In [21]:

# Cenários selecionados para avaliação exploratória
cenarios_bootstrap = [
    ("has_face", 1),
    ("video_duration_s", 31),
    ("video_duration_s", 18)
]

metricas_bootstrap = [
    "klike_score",
    "ctr_consistente",
    "engagement_rate"
]

resultados_bootstrap = []

for feature, valor_proposto in cenarios_bootstrap:
    for metrica in metricas_bootstrap:

        resultado = bootstrap_contextual(
            campanha=campanha_exemplo,
            feature=feature,
            valor_proposto=valor_proposto,
            metrica=metrica,
            n_bootstrap=2000,
            random_state=RANDOM_STATE
        )

        resultados_bootstrap.append(resultado)

tabela_bootstrap = pd.DataFrame(resultados_bootstrap)

print("Campanha:", campanha_exemplo["campaign_id"])

print("\nIntervalos bootstrap para diferenças observacionais:")
display(tabela_bootstrap.round(3))

print("\nQuantidade de comparações realizadas:")
print(len(tabela_bootstrap))

print("\nIntervalos que incluem zero:")
display(
    tabela_bootstrap["Inclui zero"]
    .value_counts()
    .rename_axis("Inclui zero")
    .reset_index(name="Comparações")
)

assert len(tabela_bootstrap) == 9

assert (
    tabela_bootstrap["Status"] == "Bootstrap concluído"
).all()

print("\nAvaliação de robustez concluída.")


Campanha: KLK-0250

Intervalos bootstrap para diferenças observacionais:


,Feature,Proposto,Métrica,Contexto,N original,N proposto,Diferença observada,IC 95% inferior,IC 95% superior,Inclui zero,Unidade,Status
0,has_face,1,Klike Score,Plataforma,30,40,13.562,6.447,21.270,False,pontos,Bootstrap concluído
1,has_face,1,CTR consistente,Plataforma,27,35,2.570,0.251,4.889,False,p.p.,Bootstrap concluído
2,has_face,1,Engagement Rate,Plataforma,29,37,0.635,-0.149,1.501,True,p.p.,Bootstrap concluído
3,video_duration_s,31,Klike Score,Plataforma,16,16,4.600,-6.244,15.457,True,pontos,Bootstrap concluído
4,video_duration_s,31,CTR consistente,Plataforma,14,14,1.893,-1.890,5.439,True,p.p.,Bootstrap concluído
5,video_duration_s,31,Engagement Rate,Plataforma,14,16,0.235,-1.281,1.789,True,p.p.,Bootstrap concluído
6,video_duration_s,18,Klike Score,Plataforma,16,16,-1.688,-14.857,10.382,True,pontos,Bootstrap concluído
7,video_duration_s,18,CTR consistente,Plataforma,14,16,-1.193,-5.015,2.573,True,p.p.,Bootstrap concluído
8,video_duration_s,18,Engagement Rate,Plataforma,14,14,-0.306,-1.701,1.057,True,p.p.,Bootstrap concluído



Quantidade de comparações realizadas:
9

Intervalos que incluem zero:


,Inclui zero,Comparações
0,True,7
1,False,2



Avaliação de robustez concluída.



---

## 6. Seleção e priorização das recomendações

### 6.1. Avaliação sistemática dos candidatos

Antes de selecionar as recomendações finais, será realizada uma avaliação de robustez de todos os cenários alternativos gerados pelo engine.

Para cada cenário, serão reunidos:

- A diferença prevista pelo Ridge reduzido.
- A diferença observacional no Klike Score.
- O intervalo bootstrap de 95% da diferença observacional.
- As evidências complementares de CTR e Engagement Rate.
- O contexto utilizado e a quantidade de observações dos grupos.

Cenários numéricos que representem a mesma comparação entre faixas de duração serão agrupados, preservando a alternativa com maior diferença prevista pelo Ridge. Essa escolha será uma regra de representação, não uma evidência de que a duração selecionada seja ideal.

A interpretação dos intervalos será descritiva:

- **Associação positiva:** intervalo inteiramente acima de zero.
- **Associação negativa:** intervalo inteiramente abaixo de zero.
- **Inconclusiva:** intervalo que inclui zero.
- **Evidência insuficiente:** comparação sem suporte amostral mínimo.

Essas classificações não representam testes causais nem correções para múltiplas comparações.

Nesta etapa, as classificações servirão para caracterizar a evidência disponível. A definição das regras de elegibilidade e priorização será realizada posteriormente.


In [22]:

def avaliar_robustez_cenarios(campanha, n_bootstrap=2000):
    """
    Avalia sistematicamente a robustez observacional
    dos cenários de uma campanha.

    Remove comparações redundantes de duração.
    Não seleciona recomendações finais.
    """

    _, simulacoes_campanha = simular_impactos(campanha)

    if simulacoes_campanha.empty:
        return pd.DataFrame(), pd.DataFrame()

    entrada = preparar_campanha(campanha).iloc[0]

    # Quartis da plataforma, calculados no treinamento
    duracoes = df_evidencias.loc[
        df_evidencias["platform"] == entrada["platform"],
        "video_duration_s"
    ].dropna()

    cortes = duracoes.quantile(
        [0.25, 0.50, 0.75]
    ).to_numpy()

    # Identificar cenários que representam a mesma comparação
    cenarios_unicos = []
    chaves_utilizadas = set()

    for _, cenario in simulacoes_campanha.iterrows():

        feature = cenario["feature_alterada"]
        valor = cenario["valor_proposto"]

        if feature == "video_duration_s":
            faixa = int(
                np.searchsorted(cortes, valor, side="left") + 1
            )
            chave = (feature, faixa)
        else:
            chave = (feature, str(valor))

        if chave in chaves_utilizadas:
            continue

        chaves_utilizadas.add(chave)
        cenarios_unicos.append(cenario)

    metricas = [
        "klike_score",
        "ctr_consistente",
        "engagement_rate"
    ]

    detalhes = []
    resumos = []

    for cenario in cenarios_unicos:

        feature = cenario["feature_alterada"]
        valor = cenario["valor_proposto"]

        resultados_metrica = {}

        for metrica in metricas:

            resultado = bootstrap_contextual(
                campanha=campanha,
                feature=feature,
                valor_proposto=valor,
                metrica=metrica,
                n_bootstrap=n_bootstrap,
                random_state=RANDOM_STATE
            )

            detalhes.append(resultado)
            resultados_metrica[metrica] = resultado

        evidencia_score = resultados_metrica["klike_score"]

        if evidencia_score["Status"] != "Bootstrap concluído":
            classificacao = "Evidência insuficiente"

        else:
            inferior = evidencia_score["IC 95% inferior"]
            superior = evidencia_score["IC 95% superior"]

            if inferior > 0:
                classificacao = "Associação positiva"
            elif superior < 0:
                classificacao = "Associação negativa"
            else:
                classificacao = "Inconclusiva"

        resumos.append({
            "Feature": feature,
            "Original": cenario["valor_original"],
            "Proposto": valor,
            "Delta Ridge": cenario["delta_score_previsto"],
            "Delta Klike observado": evidencia_score.get(
                "Diferença observada", np.nan
            ),
            "IC inferior": evidencia_score.get(
                "IC 95% inferior", np.nan
            ),
            "IC superior": evidencia_score.get(
                "IC 95% superior", np.nan
            ),
            "Contexto": evidencia_score.get(
                "Contexto", "Indisponível"
            ),
            "N original": evidencia_score.get(
                "N original", np.nan
            ),
            "N proposto": evidencia_score.get(
                "N proposto", np.nan
            ),
            "Classificação": classificacao
        })

    return (
        pd.DataFrame(resumos),
        pd.DataFrame(detalhes)
    )


In [23]:

resumo_robustez, detalhes_robustez = (
    avaliar_robustez_cenarios(
        campanha=campanha_exemplo,
        n_bootstrap=2000
    )
)

print("Campanha:", campanha_exemplo["campaign_id"])

print("\nQuantidade de cenários após remover redundâncias:")
print(len(resumo_robustez))

print("\nResumo da robustez dos candidatos:")
display(resumo_robustez.round(3))

print("\nClassificação das associações:")
display(
    resumo_robustez["Classificação"]
    .value_counts()
    .rename_axis("Classificação")
    .reset_index(name="Cenários")
)

print("\nEvidências complementares de CTR e Engagement:")
display(
    detalhes_robustez.loc[
        detalhes_robustez["Métrica"].isin([
            "CTR consistente",
            "Engagement Rate"
        ]),
        [
            "Feature",
            "Proposto",
            "Métrica",
            "Diferença observada",
            "IC 95% inferior",
            "IC 95% superior",
            "Status"
        ]
    ].round(3)
)

# Verificações de consistência
assert len(detalhes_robustez) == len(resumo_robustez) * 3

assert resumo_robustez["Feature"].isin(
    features_modificaveis
).all()

assert np.isfinite(
    resumo_robustez["Delta Ridge"].to_numpy(dtype=float)
).all()

print("\nAvaliação sistemática concluída.")


Campanha: KLK-0250

Quantidade de cenários após remover redundâncias:
7

Resumo da robustez dos candidatos:


,Feature,Original,Proposto,Delta Ridge,Delta Klike observado,IC inferior,IC superior,Contexto,N original,N proposto,Classificação
0,has_face,0,1,11.750,13.562,6.447,21.270,Plataforma,30,40,Associação positiva
1,video_duration_s,22.0,31,0.078,4.600,-6.244,15.457,Plataforma,16,16,Inconclusiva
2,video_duration_s,22.0,18,-0.035,-1.688,-14.857,10.382,Plataforma,16,16,Inconclusiva
3,format,vertical,quadrado,-1.326,14.089,-1.148,27.913,Plataforma,10,22,Inconclusiva
4,format,vertical,horizontal,-1.537,16.459,2.761,29.305,Plataforma,10,38,Associação positiva
5,text_density,medium,low,-6.584,-5.090,-13.273,3.026,Plataforma,28,23,Inconclusiva
6,text_density,medium,high,-14.035,-15.875,-25.651,-6.208,Plataforma,28,19,Associação negativa



Classificação das associações:


,Classificação,Cenários
0,Inconclusiva,4
1,Associação positiva,2
2,Associação negativa,1



Evidências complementares de CTR e Engagement:


,Feature,Proposto,Métrica,Diferença observada,IC 95% inferior,IC 95% superior,Status
1,has_face,1,CTR consistente,2.570,0.251,4.889,Bootstrap concluído
2,has_face,1,Engagement Rate,0.635,-0.149,1.501,Bootstrap concluído
4,video_duration_s,31,CTR consistente,1.893,-1.890,5.439,Bootstrap concluído
5,video_duration_s,31,Engagement Rate,0.235,-1.281,1.789,Bootstrap concluído
7,video_duration_s,18,CTR consistente,-1.193,-5.015,2.573,Bootstrap concluído
8,video_duration_s,18,Engagement Rate,-0.306,-1.701,1.057,Bootstrap concluído
10,format,quadrado,CTR consistente,NaN,NaN,NaN,Amostra insuficiente
11,format,quadrado,Engagement Rate,-0.151,-2.261,1.579,Bootstrap concluído
13,format,horizontal,CTR consistente,NaN,NaN,NaN,Amostra insuficiente
14,format,horizontal,Engagement Rate,-0.140,-2.199,1.403,Bootstrap concluído



Avaliação sistemática concluída.



### 6.2. Definição dos critérios de elegibilidade e priorização

A seleção das recomendações será realizada a partir de regras explícitas, utilizando os resultados preditivos e observacionais calculados anteriormente.

Os critérios adotados serão:

1. **Diferença preditiva positiva:** o cenário deve apresentar Klike Score previsto superior ao da configuração original.
2. **Relevância mínima:** diferenças previstas inferiores a 1 ponto serão classificadas como marginais, evitando priorizar alterações com ganhos numéricos muito pequenos.
3. **Evidência contextual favorável:** para ser elegível como recomendação, o cenário deverá apresentar associação positiva no Klike Score observacional, com intervalo bootstrap inteiramente acima de zero.
4. **Consistência das métricas:** associações negativas com CTR ou Engagement Rate, quando seus intervalos estiverem inteiramente abaixo de zero, impedirão a classificação automática como recomendação elegível.
5. **Ausência de suporte:** comparações inconclusivas poderão ser apresentadas como hipóteses para experimentação, mas não como melhorias sustentadas pelas evidências.

O limite mínimo de 1 ponto é uma **heurística operacional de relevância**, não um limiar estatístico validado. Ele será utilizado para evitar diferenças praticamente nulas, sem representar uma estimativa de incerteza do Ridge.

Os candidatos elegíveis serão ordenados pela diferença prevista no Klike Score. Quando houver várias alternativas para a mesma feature, será preservada somente a alternativa elegível com maior diferença prevista.

A classificação será exploratória, pois as associações não foram ajustadas para confundimento ou múltiplas comparações.

O sistema não criará recomendações artificiais para atingir um número fixo. Quando houver menos de três oportunidades elegíveis, essa limitação será explicitamente apresentada.


In [24]:

def priorizar_candidatos(
    resumo_robustez,
    detalhes_robustez,
    delta_minimo=1.0,
    max_recomendacoes=3
):
    """
    Classifica cenários utilizando diferenças do Ridge
    e evidências observacionais.

    Retorna:
    - avaliacao: todos os candidatos classificados.
    - priorizados: até 3 candidatos elegíveis.

    Os critérios são exploratórios, não causais.
    """

    if delta_minimo < 0:
        raise ValueError("delta_minimo não pode ser negativo.")

    if max_recomendacoes < 1:
        raise ValueError("max_recomendacoes deve ser positivo.")

    avaliacao = resumo_robustez.copy()
    classificacoes = []
    justificativas = []

    for _, candidato in avaliacao.iterrows():

        delta = float(candidato["Delta Ridge"])
        associacao = candidato["Classificação"]

        # Localizar as evidências do cenário
        evidencias = detalhes_robustez.loc[
            (detalhes_robustez["Feature"] == candidato["Feature"])
            & (
                detalhes_robustez["Proposto"]
                == candidato["Proposto"]
            )
        ]

        # Verificar associações desfavoráveis em CTR/Engagement
        metricas_desfavoraveis = []

        for _, evidencia in evidencias.iterrows():

            if evidencia["Métrica"] not in [
                "CTR consistente",
                "Engagement Rate"
            ]:
                continue

            if evidencia["Status"] != "Bootstrap concluído":
                continue

            limite_superior = evidencia["IC 95% superior"]

            if pd.notna(limite_superior) and limite_superior < 0:
                metricas_desfavoraveis.append(
                    evidencia["Métrica"]
                )

        # Classificação dos candidatos
        if delta <= 0:
            status = "Não priorizar"
            motivo = "Diferença prevista não positiva."

        elif delta < delta_minimo:
            status = "Não priorizar"
            motivo = "Diferença prevista marginal."

        elif associacao == "Associação negativa":
            status = "Não priorizar"
            motivo = "Associação observacional desfavorável."

        elif metricas_desfavoraveis:
            status = "Revisão necessária"
            motivo = (
                "Associação desfavorável em: "
                + ", ".join(metricas_desfavoraveis)
            )

        elif associacao == "Associação positiva":
            status = "Elegível"
            motivo = (
                "Delta previsto positivo e associação "
                "observacional favorável."
            )

        elif associacao == "Inconclusiva":
            status = "Teste exploratório"
            motivo = (
                "Delta previsto positivo, mas associação "
                "observacional inconclusiva."
            )

        else:
            status = "Evidência insuficiente"
            motivo = "Suporte contextual insuficiente."

        classificacoes.append(status)
        justificativas.append(motivo)

    avaliacao["Status decisão"] = classificacoes
    avaliacao["Justificativa"] = justificativas

    # Ordenar todos os candidatos
    avaliacao = avaliacao.sort_values(
        "Delta Ridge",
        ascending=False,
        kind="stable"
    ).reset_index(drop=True)

    # Selecionar somente candidatos elegíveis
    priorizados = avaliacao.loc[
        avaliacao["Status decisão"] == "Elegível"
    ].copy()

    # Evitar múltiplas recomendações para a mesma feature
    priorizados = (
        priorizados
        .drop_duplicates(subset=["Feature"], keep="first")
        .head(max_recomendacoes)
        .reset_index(drop=True)
    )

    priorizados.insert(
        0,
        "Prioridade",
        range(1, len(priorizados) + 1)
    )

    return avaliacao, priorizados


In [25]:

avaliacao_candidatos, candidatos_priorizados = (
    priorizar_candidatos(
        resumo_robustez=resumo_robustez,
        detalhes_robustez=detalhes_robustez,
        delta_minimo=1.0,
        max_recomendacoes=3
    )
)

print("Campanha:", campanha_exemplo["campaign_id"])

print("\nClassificação de todos os candidatos:")
display(
    avaliacao_candidatos[
        [
            "Feature",
            "Original",
            "Proposto",
            "Delta Ridge",
            "Classificação",
            "Status decisão",
            "Justificativa"
        ]
    ].round(3)
)

print("\nDistribuição das decisões:")
display(
    avaliacao_candidatos["Status decisão"]
    .value_counts()
    .rename_axis("Decisão")
    .reset_index(name="Quantidade")
)

print("\nCandidatos elegíveis e priorizados:")
display(
    candidatos_priorizados[
        [
            "Prioridade",
            "Feature",
            "Original",
            "Proposto",
            "Delta Ridge",
            "Contexto",
            "Classificação"
        ]
    ].round(3)
)

# Verificações
assert len(candidatos_priorizados) <= 3

assert candidatos_priorizados["Feature"].is_unique

assert (
    candidatos_priorizados["Delta Ridge"] >= 1.0
).all()

assert (
    candidatos_priorizados["Classificação"]
    == "Associação positiva"
).all()

print("\nPriorização executada com sucesso.")


Campanha: KLK-0250

Classificação de todos os candidatos:


,Feature,Original,Proposto,Delta Ridge,Classificação,Status decisão,Justificativa
0,has_face,0,1,11.750,Associação positiva,Elegível,Delta previsto positivo e associação observaci...
1,video_duration_s,22.0,31,0.078,Inconclusiva,Não priorizar,Diferença prevista marginal.
2,video_duration_s,22.0,18,-0.035,Inconclusiva,Não priorizar,Diferença prevista não positiva.
3,format,vertical,quadrado,-1.326,Inconclusiva,Não priorizar,Diferença prevista não positiva.
4,format,vertical,horizontal,-1.537,Associação positiva,Não priorizar,Diferença prevista não positiva.
5,text_density,medium,low,-6.584,Inconclusiva,Não priorizar,Diferença prevista não positiva.
6,text_density,medium,high,-14.035,Associação negativa,Não priorizar,Diferença prevista não positiva.



Distribuição das decisões:


,Decisão,Quantidade
0,Não priorizar,6
1,Elegível,1



Candidatos elegíveis e priorizados:


,Prioridade,Feature,Original,Proposto,Delta Ridge,Contexto,Classificação
0,1,has_face,0,1,11.75,Plataforma,Associação positiva



Priorização executada com sucesso.



---

## 7. Recommendations Engine — Protótipo funcional

### 7.1. Integração dos componentes

O Recommendations Engine será implementado como uma função Python que recebe os atributos de uma campanha e executa automaticamente as seguintes operações:

1. Validar e preparar a campanha utilizando as features do Ridge reduzido.
2. Gerar cenários alternativos modificando uma característica criativa por vez.
3. Calcular as diferenças previstas no Klike Score.
4. Avaliar as associações contextuais utilizando somente os dados de treinamento.
5. Investigar a robustez observacional por bootstrap.
6. Classificar os candidatos e priorizar as alterações elegíveis.
7. Produzir recomendações específicas, quantificadas e compreensíveis para profissionais de marketing.

### Estrutura do output

O sistema retornará:

- Identificador da campanha, quando disponível.
- Klike Score previsto para sua configuração original.
- Quantidade de cenários avaliados.
- Recomendações elegíveis, ordenadas pela diferença prevista no Klike Score.
- Evidências observacionais, contexto utilizado e intervalos bootstrap.
- Diagnóstico das alterações que não atenderam aos critérios.

A função aceitará campanhas existentes e novas entradas compatíveis com o modelo.

**Limitação metodológica:** diferenças previstas entre configurações não constituem efeitos causais. As associações observacionais serão apresentadas separadamente e não serão interpretadas como garantias de melhoria.

Quando houver menos de três oportunidades elegíveis, o sistema informará essa limitação em vez de gerar estimativas artificiais.


In [26]:

def descrever_acao(feature, original, proposto):
    """
    Converte a alteração de uma feature em uma
    sugestão compreensível para o time de marketing.
    """

    acoes = {
        "has_hook": "Adicionar um hook nos primeiros 3 segundos",
        "has_face": "Incluir rosto humano no criativo",
        "has_subtitle": "Adicionar legendas ao vídeo",
        "has_cta": "Adicionar uma chamada para ação (CTA)",
        "format": (
            f"Alterar o formato de {original} para {proposto}"
        ),
        "text_density": (
            f"Alterar a densidade de texto de "
            f"{original} para {proposto}"
        ),
        "video_duration_s": (
            f"Alterar a duração de {original:g}s "
            f"para {proposto:g}s"
        )
    }

    return acoes[feature]


def recomendar_campanha(
    campanha,
    max_recomendacoes=3,
    delta_minimo=1.0,
    n_bootstrap=2000
):
    """
    Recommendations Engine híbrido.

    Combina diferenças previstas pelo Ridge reduzido
    e associações contextuais obtidas no treinamento.

    Não estima efeitos causais.
    """

    # 1. Validar a entrada
    entrada = preparar_campanha(campanha)

    campanha_dict = (
        campanha.to_dict()
        if isinstance(campanha, pd.Series)
        else campanha
    )

    campaign_id = campanha_dict.get(
        "campaign_id",
        "Campanha sem identificador"
    )

    # 2. Calcular o score previsto da campanha original
    score_original = float(
        modelo_ridge.predict(entrada)[0]
    )

    # 3. Avaliar cenários e sua robustez
    resumo, detalhes = avaliar_robustez_cenarios(
        campanha=campanha,
        n_bootstrap=n_bootstrap
    )

    # 4. Tratar ausência de cenários
    if resumo.empty:
        return {
            "campaign_id": campaign_id,
            "score_original": score_original,
            "total_cenarios": 0,
            "total_recomendacoes": 0,
            "recomendacoes": [],
            "avaliacao": pd.DataFrame(),
            "evidencias": pd.DataFrame()
        }

    # 5. Aplicar os critérios de elegibilidade
    avaliacao, priorizados = priorizar_candidatos(
        resumo_robustez=resumo,
        detalhes_robustez=detalhes,
        delta_minimo=delta_minimo,
        max_recomendacoes=max_recomendacoes
    )

    recomendacoes = []

    # 6. Construir as recomendações
    for _, candidato in priorizados.iterrows():

        feature = candidato["Feature"]
        original = candidato["Original"]
        proposto = candidato["Proposto"]

        delta = float(candidato["Delta Ridge"])

        # Evidências da alteração selecionada
        evidencias_candidato = detalhes.loc[
            (detalhes["Feature"] == feature)
            & (detalhes["Proposto"] == proposto)
        ]

        evidencia_score = evidencias_candidato.loc[
            evidencias_candidato["Métrica"] == "Klike Score"
        ].iloc[0]

        # Construir a sugestão acionável
        acao = descrever_acao(
            feature,
            original,
            proposto
        )

        contexto = candidato["Contexto"]

        if contexto == "Plataforma + categoria":
            descricao_contexto = (
                f"{entrada.iloc[0]['platform']} / "
                f"{entrada.iloc[0]['category']}"
            )
        else:
            descricao_contexto = entrada.iloc[0]["platform"]

        diferenca_observada = float(
            evidencia_score["Diferença observada"]
        )

        limite_inferior = float(
            evidencia_score["IC 95% inferior"]
        )

        limite_superior = float(
            evidencia_score["IC 95% superior"]
        )

        texto = (
            f"{acao}. "
            f"O Ridge prevê uma diferença de "
            f"{delta:+.2f} pontos no Klike Score "
            f"para essa configuração alternativa. "
            f"Nas campanhas comparáveis de "
            f"{descricao_contexto}, a diferença média "
            f"observada entre os grupos foi de "
            f"{diferenca_observada:+.2f} pontos "
            f"(IC bootstrap 95%: "
            f"{limite_inferior:+.2f} a "
            f"{limite_superior:+.2f}). "
            f"Esses resultados não demonstram "
            f"uma relação causal."
        )

        recomendacoes.append({
            "prioridade": int(candidato["Prioridade"]),
            "feature": feature,
            "valor_original": original,
            "valor_proposto": proposto,
            "delta_ridge": delta,
            "contexto": descricao_contexto,
            "diferenca_observada": diferenca_observada,
            "ic_inferior": limite_inferior,
            "ic_superior": limite_superior,
            "n_original": int(evidencia_score["N original"]),
            "n_proposto": int(evidencia_score["N proposto"]),
            "texto": texto
        })

    # 7. Retornar resultado estruturado
    return {
        "campaign_id": campaign_id,
        "score_original": score_original,
        "total_cenarios": len(avaliacao),
        "total_recomendacoes": len(recomendacoes),
        "recomendacoes": recomendacoes,
        "avaliacao": avaliacao,
        "evidencias": detalhes
    }


In [27]:

# Executar o Recommendations Engine
resultado_engine = recomendar_campanha(
    campanha=campanha_exemplo,
    max_recomendacoes=3,
    delta_minimo=1.0,
    n_bootstrap=2000
)

print("=" * 65)
print("KLIKE — RECOMMENDATIONS ENGINE")
print("=" * 65)

print("\nCampanha:", resultado_engine["campaign_id"])

print(
    "Klike Score previsto:",
    f"{resultado_engine['score_original']:.3f}"
)

print(
    "Cenários avaliados:",
    resultado_engine["total_cenarios"]
)

print(
    "Recomendações elegíveis:",
    resultado_engine["total_recomendacoes"]
)

print("\nRECOMENDAÇÕES")
print("-" * 65)

if resultado_engine["recomendacoes"]:

    for recomendacao in resultado_engine["recomendacoes"]:

        print(
            f"\n{recomendacao['prioridade']}ª RECOMENDAÇÃO"
        )

        print(recomendacao["texto"])

        print(
            "Amostra contextual:",
            f"N={recomendacao['n_original']} vs "
            f"N={recomendacao['n_proposto']}"
        )

else:
    print(
        "Nenhuma alteração atendeu aos critérios "
        "de elegibilidade estabelecidos."
    )

print("\nDIAGNÓSTICO")
print("-" * 65)

display(
    resultado_engine["avaliacao"][
        [
            "Feature",
            "Original",
            "Proposto",
            "Delta Ridge",
            "Status decisão"
        ]
    ].round(3)
)

# Verificações
assert resultado_engine["total_recomendacoes"] <= 3

assert resultado_engine["total_cenarios"] >= (
    resultado_engine["total_recomendacoes"]
)

assert all(
    recomendacao["delta_ridge"] >= 1.0
    for recomendacao in resultado_engine["recomendacoes"]
)

print("\nExecução concluída.")


KLIKE — RECOMMENDATIONS ENGINE

Campanha: KLK-0250
Klike Score previsto: 66.506
Cenários avaliados: 7
Recomendações elegíveis: 1

RECOMENDAÇÕES
-----------------------------------------------------------------

1ª RECOMENDAÇÃO
Incluir rosto humano no criativo. O Ridge prevê uma diferença de +11.75 pontos no Klike Score para essa configuração alternativa. Nas campanhas comparáveis de LinkedIn, a diferença média observada entre os grupos foi de +13.56 pontos (IC bootstrap 95%: +6.45 a +21.27). Esses resultados não demonstram uma relação causal.
Amostra contextual: N=30 vs N=40

DIAGNÓSTICO
-----------------------------------------------------------------


,Feature,Original,Proposto,Delta Ridge,Status decisão
0,has_face,0,1,11.750,Elegível
1,video_duration_s,22.0,31,0.078,Não priorizar
2,video_duration_s,22.0,18,-0.035,Não priorizar
3,format,vertical,quadrado,-1.326,Não priorizar
4,format,vertical,horizontal,-1.537,Não priorizar
5,text_density,medium,low,-6.584,Não priorizar
6,text_density,medium,high,-14.035,Não priorizar



Execução concluída.
